A. SQL AND DATA


1. Connecting to our Google account and running a query to confirm we have access.

In [ ]:
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()
project_id = 'dahcc-sepsis'
client = bigquery.Client(project=project_id)

query = """
SELECT COUNT(*) as n_patients
FROM `physionet-data.mimiciii_clinical.patients`
"""
df = client.query(query).to_dataframe()
print(df)

2. Here we start our actual query to identify the sepsis patients using the ICD codes.

In [ ]:
sepsis_query = """
SELECT DISTINCT
  d.subject_id,
  d.hadm_id,
  a.hospital_expire_flag AS mortality,
  a.admittime,
  a.dischtime,
  p.gender,
  DATE_DIFF(DATE(a.admittime), DATE(p.dob), YEAR) AS age
FROM `physionet-data.mimiciii_clinical.diagnoses_icd` d
JOIN `physionet-data.mimiciii_clinical.admissions` a
  ON d.hadm_id = a.hadm_id
JOIN `physionet-data.mimiciii_clinical.patients` p
  ON d.subject_id = p.subject_id
WHERE d.icd9_code IN ('99591', '99592', '78552')  -- sepsis ICD-9 codes
  AND DATE_DIFF(DATE(a.admittime), DATE(p.dob), YEAR) >= 18  -- adults only
"""

cohort = client.query(sepsis_query).to_dataframe()
print(f"Sepsis patients: {len(cohort)}")
print(f"Mortality rate: {cohort['mortality'].mean():.1%}")




3. A check to know if some patients have multiple admissions.

In [ ]:
print(f"Total rows: {len(cohort)}")
print(f"Unique patients (subject_id): {cohort['subject_id'].nunique()}")
print(f"Unique admissions (hadm_id): {cohort['hadm_id'].nunique()}")

# Age sanity check
print(f"\nAge stats:")
print(cohort['age'].describe())

4. A check to see how many ICU stays per admission.

In [ ]:
initial_icu_query = """
WITH sepsis_hadm AS (
  SELECT DISTINCT hadm_id
  FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
  WHERE icd9_code IN ('99591', '99592', '78552')
)
SELECT
  COUNT(*) AS total_icu_stays,
  COUNT(DISTINCT i.hadm_id) AS unique_admissions
FROM sepsis_hadm s
JOIN `physionet-data.mimiciii_clinical.icustays` i ON s.hadm_id = i.hadm_id
"""
result = client.query(initial_icu_query).to_dataframe()
print(result)
print(f"\nExtra ICU stays: {int(result['total_icu_stays'][0]) - int(result['unique_admissions'][0])}")

5. Three data issues found from the checks above:

(A) Multiple admissions per patient: 4,683 unique patients across 5,319 admissions. 636 patients had more than one admission. We keep only the first admission per patient.

(B) Age values above 90: MIMIC-III shifts ages above 90 to 300+ to protect patient privacy. We cap all ages at 90.

(C) Multiple ICU stays per admission: some admissions link to more than one ICU stay. We use ROW_NUMBER() to keep only the first ICU stay per admission.

In [ ]:
cohort_icu_query = """
WITH sepsis_admissions AS (
  SELECT DISTINCT
    d.subject_id, d.hadm_id,
    a.hospital_expire_flag AS mortality,
    a.admittime, p.gender,
    DATE_DIFF(DATE(a.admittime), DATE(p.dob), YEAR) AS age
  FROM `physionet-data.mimiciii_clinical.diagnoses_icd` d
  JOIN `physionet-data.mimiciii_clinical.admissions` a ON d.hadm_id = a.hadm_id
  JOIN `physionet-data.mimiciii_clinical.patients` p ON d.subject_id = p.subject_id
  WHERE d.icd9_code IN ('99591', '99592', '78552')
    AND DATE_DIFF(DATE(a.admittime), DATE(p.dob), YEAR) >= 18
),
ranked_icu AS (
  SELECT s.*, i.icustay_id, i.intime AS icu_intime, i.outtime AS icu_outtime,
    DATETIME_DIFF(i.outtime, i.intime, HOUR) / 24.0 AS icu_los_days,
    ROW_NUMBER() OVER (PARTITION BY s.hadm_id ORDER BY i.intime) AS icu_rank  -- rank ICU stays per admission
  FROM sepsis_admissions s
  JOIN `physionet-data.mimiciii_clinical.icustays` i ON s.hadm_id = i.hadm_id
)
SELECT * EXCEPT(icu_rank)
FROM ranked_icu
WHERE icu_rank = 1  -- keep first ICU stay only
ORDER BY subject_id, icu_intime
"""

cohort_icu = client.query(cohort_icu_query).to_dataframe()
print(f"ICU stays after dedup: {len(cohort_icu)}")


In [ ]:
# Cap age at 90
cohort_icu['age'] = cohort_icu['age'].clip(upper=90)

# Keep only the first admission per patient
cohort_final = (
    cohort_icu
    .sort_values('icu_intime')
    .drop_duplicates(subset='subject_id', keep='first')
    .reset_index(drop=True)
)

# Quick checks
print(f"After age cap, patients with age > 200: {(cohort_icu['age'] > 200).sum()}")  # must be 0
print(f"Final cohort: {len(cohort_final)} unique patients")
print(f"Mortality rate: {cohort_final['mortality'].mean():.1%}")
print(f"\nAge distribution (capped):")
print(cohort_final['age'].describe().round(1))
print(f"\nGender split:\n{cohort_final['gender'].value_counts()}")
print(f"\nICU LOS (days):\n{cohort_final['icu_los_days'].describe().round(1)}")

short_stays = cohort_final[cohort_final['icu_los_days'] < (4/24)]
print(f"\nICU stays under 4 hours: {len(short_stays)}")

# Remove short stays
cohort_final = cohort_final[cohort_final['icu_los_days'] >= (4/24)].reset_index(drop=True)
print(f"Cohort after removing short stays: {len(cohort_final)}")

6. Vital signs from CHARTEVENTS, first 24 hours of each ICU stay.

In [ ]:
vitals_query = """
WITH vital_items AS (
  -- item IDs for each vital sign (CareVue and MetaVision)
  SELECT itemid, vital
  FROM UNNEST([
    STRUCT(211 AS itemid, 'heart_rate' AS vital), STRUCT(220045, 'heart_rate'),
    STRUCT(646, 'spo2'),   STRUCT(220277, 'spo2'),
    STRUCT(618, 'resp_rate'), STRUCT(220210, 'resp_rate'), STRUCT(3603, 'resp_rate'),
    STRUCT(51, 'sbp'),    STRUCT(220179, 'sbp'), STRUCT(455, 'sbp'),
    STRUCT(52, 'map'),    STRUCT(220052, 'map'), STRUCT(456, 'map'),
    STRUCT(676, 'temp_c'), STRUCT(677, 'temp_c'),
    STRUCT(678, 'temp_f'), STRUCT(679, 'temp_f'),
    STRUCT(223762, 'temp_c'), STRUCT(223761, 'temp_f')
  ])
),
raw_vitals AS (
  SELECT
    c.icustay_id,
    v.vital,
    CASE
      WHEN v.vital = 'temp_f' THEN (c.valuenum - 32) * 5.0/9  -- convert Fahrenheit to Celsius
      ELSE c.valuenum
    END AS value
  FROM `physionet-data.mimiciii_clinical.chartevents` c
  JOIN vital_items v ON c.itemid = v.itemid
  JOIN `physionet-data.mimiciii_clinical.icustays` i ON c.icustay_id = i.icustay_id
  WHERE c.charttime BETWEEN i.intime AND DATETIME_ADD(i.intime, INTERVAL 24 HOUR)  -- first 24 hours
    AND c.valuenum IS NOT NULL
    AND c.error IS DISTINCT FROM 1
    -- plausibility bounds
    AND NOT (v.vital = 'heart_rate' AND (c.valuenum < 0   OR c.valuenum > 300))
    AND NOT (v.vital = 'spo2'       AND (c.valuenum < 50  OR c.valuenum > 100))
    AND NOT (v.vital = 'resp_rate'  AND (c.valuenum < 0   OR c.valuenum > 80))
    AND NOT (v.vital = 'sbp'        AND (c.valuenum < 0   OR c.valuenum > 300))
    AND NOT (v.vital = 'map'        AND (c.valuenum < 0   OR c.valuenum > 200))
    AND NOT (v.vital = 'temp_c'     AND (c.valuenum < 25  OR c.valuenum > 45))
    AND NOT (v.vital = 'temp_f'     AND (c.valuenum < 77  OR c.valuenum > 113))
)
SELECT
  icustay_id,
  AVG(CASE WHEN vital = 'heart_rate' THEN value END) AS hr_mean,
  MIN(CASE WHEN vital = 'heart_rate' THEN value END) AS hr_min,
  MAX(CASE WHEN vital = 'heart_rate' THEN value END) AS hr_max,
  AVG(CASE WHEN vital = 'spo2' THEN value END)       AS spo2_mean,
  MIN(CASE WHEN vital = 'spo2' THEN value END)       AS spo2_min,
  AVG(CASE WHEN vital = 'resp_rate' THEN value END)  AS rr_mean,
  MAX(CASE WHEN vital = 'resp_rate' THEN value END)  AS rr_max,
  AVG(CASE WHEN vital = 'sbp' THEN value END)        AS sbp_mean,
  MIN(CASE WHEN vital = 'sbp' THEN value END)        AS sbp_min,
  AVG(CASE WHEN vital = 'map' THEN value END)        AS map_mean,
  MIN(CASE WHEN vital = 'map' THEN value END)        AS map_min,
  AVG(CASE WHEN vital IN ('temp_c','temp_f') THEN value END) AS temp_mean,
  MIN(CASE WHEN vital IN ('temp_c','temp_f') THEN value END) AS temp_min,
  MAX(CASE WHEN vital IN ('temp_c','temp_f') THEN value END) AS temp_max
FROM raw_vitals
GROUP BY icustay_id
"""

vitals = client.query(vitals_query).to_dataframe()
print(f"ICU stays with vitals: {len(vitals)}")
print(f"\nMissing values per vital:")
print(vitals.isnull().sum())
print(f"\nVitals summary:")
print(vitals.describe().round(1))


7. labs query

In [ ]:
labs_query = """
WITH lab_items AS (
  -- item IDs for each lab test
  SELECT itemid, lab
  FROM UNNEST([
    STRUCT(50813 AS itemid, 'lactate' AS lab),
    STRUCT(50912, 'creatinine'),
    STRUCT(50885, 'bilirubin'),
    STRUCT(51265, 'platelets'),
    STRUCT(51301, 'wbc'),
    STRUCT(51222, 'hemoglobin'),
    STRUCT(50983, 'sodium'),
    STRUCT(50971, 'potassium'),
    STRUCT(51006, 'bun'),
    STRUCT(50931, 'glucose'),
    STRUCT(50820, 'ph'),
    STRUCT(50821, 'pao2')
  ])
),
cohort_stays AS (
  SELECT DISTINCT icustay_id, icu_intime
  FROM `dahcc-sepsis.sepsis_cohort.cohort_final`
),
raw_labs AS (
  SELECT
    cs.icustay_id,
    li.lab,
    le.valuenum AS value
  FROM `physionet-data.mimiciii_clinical.labevents` le
  JOIN lab_items li ON le.itemid = li.itemid
  JOIN `physionet-data.mimiciii_clinical.icustays` i
    ON le.hadm_id = i.hadm_id
  JOIN cohort_stays cs ON i.icustay_id = cs.icustay_id
  WHERE le.charttime BETWEEN i.intime
    AND DATETIME_ADD(i.intime, INTERVAL 24 HOUR)  -- first 24 hours
    AND le.valuenum IS NOT NULL
    -- plausibility bounds
    AND NOT (li.lab = 'lactate'    AND (le.valuenum < 0   OR le.valuenum > 30))
    AND NOT (li.lab = 'creatinine' AND (le.valuenum < 0   OR le.valuenum > 50))
    AND NOT (li.lab = 'bilirubin'  AND (le.valuenum < 0   OR le.valuenum > 100))
    AND NOT (li.lab = 'platelets'  AND (le.valuenum < 0   OR le.valuenum > 2000))
    AND NOT (li.lab = 'wbc'        AND (le.valuenum < 0   OR le.valuenum > 500))
    AND NOT (li.lab = 'ph'         AND (le.valuenum < 6.5 OR le.valuenum > 8.0))
    AND NOT (li.lab = 'glucose'    AND (le.valuenum < 0   OR le.valuenum > 2000))
)
SELECT
  icustay_id,
  AVG(CASE WHEN lab = 'lactate'    THEN value END) AS lactate_mean,
  MAX(CASE WHEN lab = 'lactate'    THEN value END) AS lactate_max,
  AVG(CASE WHEN lab = 'creatinine' THEN value END) AS creatinine_mean,
  MAX(CASE WHEN lab = 'creatinine' THEN value END) AS creatinine_max,
  AVG(CASE WHEN lab = 'bilirubin'  THEN value END) AS bilirubin_mean,
  AVG(CASE WHEN lab = 'platelets'  THEN value END) AS platelets_mean,
  MIN(CASE WHEN lab = 'platelets'  THEN value END) AS platelets_min,
  AVG(CASE WHEN lab = 'wbc'        THEN value END) AS wbc_mean,
  AVG(CASE WHEN lab = 'hemoglobin' THEN value END) AS hemoglobin_mean,
  AVG(CASE WHEN lab = 'sodium'     THEN value END) AS sodium_mean,
  AVG(CASE WHEN lab = 'potassium'  THEN value END) AS potassium_mean,
  AVG(CASE WHEN lab = 'bun'        THEN value END) AS bun_mean,
  MAX(CASE WHEN lab = 'bun'        THEN value END) AS bun_max,
  AVG(CASE WHEN lab = 'glucose'    THEN value END) AS glucose_mean,
  AVG(CASE WHEN lab = 'ph'         THEN value END) AS ph_mean,
  MIN(CASE WHEN lab = 'ph'         THEN value END) AS ph_min,
  AVG(CASE WHEN lab = 'pao2'       THEN value END) AS pao2_mean,
  MIN(CASE WHEN lab = 'pao2'       THEN value END) AS pao2_min
FROM raw_labs
GROUP BY icustay_id
"""

labs = client.query(labs_query).to_dataframe()
print(f"ICU stays with labs: {len(labs)}")
print(f"\nMissing values per lab:")
print(labs.isnull().sum())
print(f"\nLabs summary:")
print(labs.describe().round(2))


8. Merge vitals and labs into one dataframe.

In [ ]:
import pandas as pd

# Start from the cleaned cohort
df = cohort_final.copy()  # 4676 rows, has icustay_id

# Merge vitals
df = df.merge(vitals, on='icustay_id', how='left')
print(f"After merging vitals: {len(df)} rows")

# Merge labs
df = df.merge(labs, on='icustay_id', how='left')
print(f"After merging labs: {len(df)} rows")

# Feature and missing value summary
print(f"\nTotal features: {df.shape[1]}")
print(f"\nMissing value summary:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
print(pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
      .query('missing_count > 0')
      .sort_values('missing_%', ascending=False))

9. Handle Missing Values.

In [ ]:
from sklearn.impute import SimpleImputer
import numpy as np

# Drop identifiers and target, keep features
drop_cols = ['subject_id', 'hadm_id', 'icustay_id',
             'admittime', 'icu_intime', 'icu_outtime', 'mortality']
feature_cols = [c for c in df.columns if c not in drop_cols]

# Encode gender
df['gender_bin'] = (df['gender'] == 'M').astype(int)
feature_cols = [c for c in feature_cols if c != 'gender'] + ['gender_bin']

X = df[feature_cols]
y = df['mortality']

# Median imputation for skewed lab values
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=feature_cols
)
print(f"Feature matrix shape: {X_imputed.shape}")
print(f"Class balance, mortality rate: {y.mean():.1%}")
# Check temperature range
print(f"\nTemp mean range: {X_imputed['temp_mean'].min():.1f}°C to {X_imputed['temp_mean'].max():.1f}°C")

B. EDA and VISUALIZATION

1. Mortality Overview


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Cohort Overview', fontsize=14, fontweight='bold')

# Mortality count
mortality_counts = y.value_counts()
axes[0].bar(['Survived', 'Died'], mortality_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Mortality Distribution')
axes[0].set_ylabel('Number of Patients')
for i, v in enumerate(mortality_counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Gender split
gender_mort = df.groupby('gender')['mortality'].mean()
axes[1].bar(['Female', 'Male'], gender_mort.values, color=['orchid', 'steelblue'])
axes[1].set_title('Mortality Rate by Gender')
axes[1].set_ylabel('Mortality Rate')
axes[1].set_ylim(0, 0.5)
for i, v in enumerate(gender_mort.values):
    axes[1].text(i, v + 0.01, f'{v:.1%}', ha='center', fontweight='bold')

# Age distribution by outcome
survived = df[df['mortality'] == 0]['age']
died = df[df['mortality'] == 1]['age']
axes[2].hist(survived, bins=20, alpha=0.6, color='steelblue', label='Survived')
axes[2].hist(died, bins=20, alpha=0.6, color='tomato', label='Died')
axes[2].set_title('Age Distribution by Outcome')
axes[2].set_xlabel('Age (years)')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Overall mortality rate: {y.mean():.1%}")
print(f"Median age - survived: {survived.median():.0f}  |  died: {died.median():.0f}")

2. ICU length of stay.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('ICU Length of Stay', fontsize=14, fontweight='bold')

# LOS distribution
axes[0].hist(df['icu_los_days'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('ICU LOS Distribution')
axes[0].set_xlabel('Days')
axes[0].set_ylabel('Count')
axes[0].set_yscale('log')

# LOS by outcome
sns.boxplot(x=y, y=df['icu_los_days'], ax=axes[1],
            palette={0: 'steelblue', 1: 'tomato'}, hue=y, legend=False)
axes[1].set_title('ICU LOS by Outcome')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Survived', 'Died'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Days')
axes[1].set_ylim(0, 40)

plt.tight_layout()
plt.show()

print(f"Median LOS - survived: {df[df['mortality']==0]['icu_los_days'].median():.1f} days")
print(f"Median LOS - died:     {df[df['mortality']==1]['icu_los_days'].median():.1f} days")

3. Key Lab distributions by outcome.

In [ ]:
key_labs = ['lactate_mean', 'creatinine_mean', 'bilirubin_mean',
            'platelets_mean', 'wbc_mean', 'ph_mean']
lab_labels = ['Lactate (mmol/L)', 'Creatinine (mg/dL)', 'Bilirubin (mg/dL)',
              'Platelets (K/uL)', 'WBC (K/uL)', 'pH']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Key Lab Values by Outcome (first 24h ICU)', fontsize=14, fontweight='bold')
axes = axes.flatten()

for ax, col, label in zip(axes, key_labs, lab_labels):
    data_survived = X_imputed.loc[y == 0, col]
    data_died = X_imputed.loc[y == 1, col]
    ax.hist(data_survived, bins=30, alpha=0.6, color='steelblue', label='Survived', density=True)
    ax.hist(data_died, bins=30, alpha=0.6, color='tomato', label='Died', density=True)
    ax.set_title(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    # Add median lines
    ax.axvline(data_survived.median(), color='steelblue', linestyle='--', linewidth=1.5)
    ax.axvline(data_died.median(), color='tomato', linestyle='--', linewidth=1.5)

plt.tight_layout()
plt.show()

4. Vital lab distributions by Outcome.


In [ ]:
key_vitals = ['hr_mean', 'sbp_mean', 'map_mean', 'rr_mean', 'spo2_mean']
vital_labels = ['Heart Rate (bpm)', 'Systolic BP (mmHg)', 'MAP (mmHg)',
                'Resp Rate (breaths/min)', 'SpO2 (%)']

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Vital Signs by Outcome (first 24h ICU)', fontsize=14, fontweight='bold')

for ax, col, label in zip(axes, key_vitals, vital_labels):
    sns.boxplot(x=y, y=X_imputed[col], ax=ax,
                palette={0: 'steelblue', 1: 'tomato'}, hue=y, legend=False)
    ax.set_title(label)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Survived', 'Died'], fontsize=8)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

5. Correlation HeatMap

In [ ]:
# Select key features for the heatmap
heatmap_cols = ['age', 'icu_los_days', 'hr_mean', 'sbp_mean', 'map_mean',
                'rr_mean', 'spo2_mean', 'lactate_mean', 'creatinine_mean',
                'bilirubin_mean', 'platelets_mean', 'wbc_mean', 'ph_mean',
                'pao2_mean', 'bun_mean', 'glucose_mean']

# Add mortality to the dataframe for correlation
heatmap_df = X_imputed[heatmap_cols].copy()
heatmap_df['mortality'] = y.values

corr = heatmap_df.corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 7}, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (including mortality)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Top features correlated with mortality
print("Top features correlated with mortality:")
print(corr['mortality'].drop('mortality').abs()
      .sort_values(ascending=False).head(10).round(3))

6. Missing Data Visualization.

In [ ]:
missing_pct = (df[feature_cols].isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(missing_pct.index, missing_pct.values,
               color=['tomato' if v > 30 else 'orange' if v > 10 else 'steelblue'
                      for v in missing_pct.values])
ax.set_xlabel('Missing (%)')
ax.set_title('Missing Data by Feature', fontweight='bold')
ax.axvline(30, color='tomato', linestyle='--', linewidth=1, label='30% threshold')
ax.axvline(10, color='orange', linestyle='--', linewidth=1, label='10% threshold')
ax.legend()
plt.tight_layout()
plt.show()

C. Machine Learning.

1. Train/Validation/Test Set split


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Step 1: hold out 15% as final test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_imputed, y, test_size=0.15, random_state=42, stratify=y
)

# Step 2: split the remaining 85% into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1765, random_state=42, stratify=y_train_val
)

# Final proportions: 70% train, 15% val, 15% test

print(f"Train:      {len(X_train)} patients ({y_train.mean():.1%} mortality)")
print(f"Validation: {len(X_val)} patients ({y_val.mean():.1%} mortality)")
print(f"Test:       {len(X_test)} patients ({y_test.mean():.1%} mortality)")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

2. Helper function for evaluation

In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve)

def evaluate_model(name, model, X_val, y_val, proba=True):
    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)[:, 1] if proba else None

    acc  = accuracy_score(y_val, y_pred)
    auc  = roc_auc_score(y_val, y_prob) if proba else None
    report = classification_report(y_val, y_pred,
                                   target_names=['Survived', 'Died'],
                                   output_dict=True)

    sensitivity = report['Died']['recall']
    specificity = report['Survived']['recall']
    f1_died     = report['Died']['f1-score']

    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(f"  Accuracy:    {acc:.3f}")
    print(f"  ROC-AUC:     {auc:.3f}" if auc else "  ROC-AUC:     N/A")
    print(f"  Sensitivity: {sensitivity:.3f}  (recall for Died)")
    print(f"  Specificity: {specificity:.3f}  (recall for Survived)")
    print(f"  F1 (Died):   {f1_died:.3f}")
    print(f"\n{classification_report(y_val, y_pred, target_names=['Survived','Died'])}")

    return {
        'name': name, 'accuracy': acc, 'roc_auc': auc,
        'sensitivity': sensitivity, 'specificity': specificity,
        'f1_died': f1_died, 'y_prob': y_prob, 'y_pred': y_pred
    }

3. Model Training, Cross-Validation and Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from scipy.stats import loguniform, uniform

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos

# Logistic Regression
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train_scaled, y_train)
lr_cv = cross_val_score(lr_pipe, X_imputed, y, cv=cv5, scoring='roc_auc', n_jobs=-1)
lr_results = evaluate_model('Logistic Regression', lr, X_val_scaled, y_val)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_cv = cross_val_score(rf, X_imputed, y, cv=cv5, scoring='roc_auc', n_jobs=-1)
rf_results = evaluate_model('Random Forest', rf, X_val, y_val)

# XGBoost, default then tuned
xgb = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    scale_pos_weight=spw, random_state=42,
    eval_metric='logloss', verbosity=0
)
xgb.fit(X_train, y_train)
xgb_results = evaluate_model('XGBoost (Default)', xgb, X_val, y_val)

param_dist = {
    'n_estimators':     [200, 300, 400, 500],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    loguniform(0.01, 0.2),
    'subsample':        uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'min_child_weight': [1, 3, 5],
    'gamma':            uniform(0, 0.3),
}
rs = RandomizedSearchCV(
    XGBClassifier(scale_pos_weight=spw, random_state=42,
                  eval_metric='logloss', verbosity=0),
    param_dist, n_iter=50, scoring='roc_auc',
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    random_state=42, n_jobs=-1, verbose=1
)
rs.fit(X_train, y_train)
tuned_xgb = rs.best_estimator_
tuned_results = evaluate_model('XGBoost (Tuned)', tuned_xgb, X_val, y_val)

# Summary
print('\n5-Fold CV AUC:')
print('=' * 55)
print(f'Logistic Regression        {lr_cv.mean():.3f} ± {lr_cv.std():.3f}')
print(f'Random Forest              {rf_cv.mean():.3f} ± {rf_cv.std():.3f}')
print(f'XGBoost tuned (3-fold CV): {rs.best_score_:.3f}')
print(f'\nXGBoost default → tuned (val AUC): '
      f'{xgb_results["roc_auc"]:.3f} → {tuned_results["roc_auc"]:.3f} '
      f'(+{tuned_results["roc_auc"] - xgb_results["roc_auc"]:.3f})')


4. Comparison Table.

In [ ]:
results_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ['y_prob', 'y_pred']}
    for r in [lr_results, rf_results, xgb_results, tuned_results]
]).set_index('name').round(3)

print('\nModel Comparison on Validation Set:')
print(results_df.to_string())


5. ROC Curves


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for results, color in zip([lr_results, rf_results, xgb_results],
                           ['steelblue', 'forestgreen', 'tomato']):
    fpr, tpr, _ = roc_curve(y_val, results['y_prob'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{results['name']} (AUC = {results['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curves, Validation Set', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

6. Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices, Validation Set', fontweight='bold')

for ax, results in zip(axes, [lr_results, rf_results, xgb_results]):
    cm = confusion_matrix(y_val, results['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Survived', 'Died'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(results['name'])

plt.tight_layout()
plt.show()

7. Feature Importance


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Feature Importance, Top 15 Features', fontweight='bold')

# Random Forest
rf_imp = pd.Series(rf.feature_importances_, index=feature_cols).nlargest(15)
rf_imp.sort_values().plot(kind='barh', ax=axes[0], color='forestgreen')
axes[0].set_title('Random Forest')
axes[0].set_xlabel('Importance')

# XGBoost
xgb_imp = pd.Series(xgb.feature_importances_, index=feature_cols).nlargest(15)
xgb_imp.sort_values().plot(kind='barh', ax=axes[1], color='tomato')
axes[1].set_title('XGBoost')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

D. Model Improvement and Evaluation

1. Threshold Optimisation

In [ ]:
from sklearn.metrics import precision_recall_curve

y_prob_val_tuned = tuned_xgb.predict_proba(X_val)[:, 1]

# Find optimal threshold using Youden's J
fpr_t, tpr_t, thresholds_roc = roc_curve(y_val, y_prob_val_tuned)
youden_j = tpr_t - fpr_t
best_idx = youden_j.argmax()
opt_threshold = thresholds_roc[best_idx]

print(f'Optimal threshold (Youden\'s J): {opt_threshold:.3f}')
print(f'  Sensitivity: {tpr_t[best_idx]:.3f}')
print(f'  Specificity: {1 - fpr_t[best_idx]:.3f}')

y_pred_opt = (y_prob_val_tuned >= opt_threshold).astype(int)
print(f'\nClassification report at optimal threshold ({opt_threshold:.3f}):')
print(classification_report(y_val, y_pred_opt, target_names=['Survived', 'Died']))

# Plots
precision_pr, recall_pr, thresholds_pr = precision_recall_curve(y_val, y_prob_val_tuned)
f1_pr = 2 * precision_pr * recall_pr / (precision_pr + recall_pr + 1e-9)
best_pr_idx = f1_pr.argmax()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sensitivity and specificity by threshold
axes[0].plot(thresholds_roc, tpr_t, label='Sensitivity', color='tomato')
axes[0].plot(thresholds_roc, 1 - fpr_t, label='Specificity', color='steelblue')
axes[0].plot(thresholds_roc, youden_j, label="Youden's J", color='purple', linestyle='--')
axes[0].axvline(opt_threshold, color='black', linestyle=':', linewidth=1.5,
                label=f'Optimal: {opt_threshold:.3f}')
axes[0].set_xlabel('Threshold')
axes[0].set_title("Sensitivity / Specificity vs Threshold", fontweight='bold')
axes[0].legend()

# Precision-recall curve
axes[1].plot(recall_pr, precision_pr, color='darkorange', linewidth=2)
axes[1].scatter(recall_pr[best_pr_idx], precision_pr[best_pr_idx],
                color='black', zorder=5,
                label=f'Best F1 threshold: {thresholds_pr[best_pr_idx]:.3f}')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

2. Model Calibration

In [ ]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss

# Platt scaling, fitted on the validation set
calibrated_xgb = CalibratedClassifierCV(FrozenEstimator(tuned_xgb), method='sigmoid')
calibrated_xgb.fit(X_val, y_val)

y_prob_test_raw  = tuned_xgb.predict_proba(X_test)[:, 1]
y_prob_test_cal  = calibrated_xgb.predict_proba(X_test)[:, 1]

prob_true_raw, prob_pred_raw = calibration_curve(y_test, y_prob_test_raw, n_bins=10)
prob_true_cal, prob_pred_cal = calibration_curve(y_test, y_prob_test_cal, n_bins=10)

brier_raw = brier_score_loss(y_test, y_prob_test_raw)
brier_cal = brier_score_loss(y_test, y_prob_test_cal)
auc_cal   = roc_auc_score(y_test, y_prob_test_cal)

print(f'Brier score uncalibrated: {brier_raw:.4f}')
print(f'Brier score calibrated:   {brier_cal:.4f}  (lower = better)')
print(f'ROC-AUC calibrated (test): {auc_cal:.3f}')

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(prob_pred_raw, prob_true_raw, 's-', color='tomato',
        label=f'Uncalibrated (Brier={brier_raw:.3f})')
ax.plot(prob_pred_cal, prob_true_cal, 's-', color='steelblue',
        label=f'Calibrated / Platt (Brier={brier_cal:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (Died)')
ax.set_title('Reliability Diagram, XGBoost Test Set', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


3. SHAP Explainability

In [ ]:
import shap

explainer = shap.TreeExplainer(tuned_xgb)
shap_values = explainer.shap_values(X_val)

# Global feature importance
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val, feature_names=feature_cols,
                  plot_type='dot', show=False)
plt.title('SHAP Summary, XGBoost Tuned', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# Mean SHAP values per feature
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_values, X_val, feature_names=feature_cols,
                  plot_type='bar', show=False)
plt.title('Mean |SHAP| Feature Importance', fontweight='bold', pad=12)
plt.tight_layout()
plt.show()


E. Clinical Baseline and Final Summary

1. SOFA Score, Clinical Baseline

SOFA score has six components: Respiratory, Coagulation, Liver, Cardiovascular, CNS, and Renal. GCS and FiO2 needed two extra BigQuery queries. The rest came from the labs and vitals already pulled. Cardiovascular is based on MAP only since vasopressor data was not extracted.

In [ ]:
gcs_query = """
WITH gcs_items AS (
  SELECT itemid, component FROM UNNEST([
    STRUCT(184  AS itemid, 'eyes'   AS component), STRUCT(220739, 'eyes'),
    STRUCT(454,  'verbal'), STRUCT(223900, 'verbal'),
    STRUCT(198,  'motor'),  STRUCT(223901, 'motor')
  ])
),
cohort_stays AS (
  SELECT DISTINCT icustay_id FROM `dahcc-sepsis.sepsis_cohort.cohort_final`
),
raw_gcs AS (
  SELECT cs.icustay_id, gi.component, c.valuenum
  FROM `physionet-data.mimiciii_clinical.chartevents` c
  JOIN gcs_items gi ON c.itemid = gi.itemid
  JOIN `physionet-data.mimiciii_clinical.icustays` i ON c.icustay_id = i.icustay_id
  JOIN cohort_stays cs ON c.icustay_id = cs.icustay_id
  WHERE c.charttime BETWEEN i.intime AND DATETIME_ADD(i.intime, INTERVAL 24 HOUR)
    AND c.valuenum IS NOT NULL AND c.error IS DISTINCT FROM 1
)
SELECT icustay_id,
  MIN(CASE WHEN component='eyes'   THEN valuenum END) +
  MIN(CASE WHEN component='verbal' THEN valuenum END) +
  MIN(CASE WHEN component='motor'  THEN valuenum END) AS gcs_min
FROM raw_gcs GROUP BY icustay_id HAVING gcs_min IS NOT NULL
"""

fio2_query = """
WITH fio2_items AS (
  SELECT itemid FROM UNNEST([3420, 190, 223835, 3422]) AS itemid
),
cohort_stays AS (
  SELECT DISTINCT icustay_id FROM `dahcc-sepsis.sepsis_cohort.cohort_final`
)
SELECT cs.icustay_id, MIN(c.valuenum) AS fio2_min
FROM `physionet-data.mimiciii_clinical.chartevents` c
JOIN fio2_items ON c.itemid = fio2_items.itemid
JOIN `physionet-data.mimiciii_clinical.icustays` i ON c.icustay_id = i.icustay_id
JOIN cohort_stays cs ON c.icustay_id = cs.icustay_id
WHERE c.charttime BETWEEN i.intime AND DATETIME_ADD(i.intime, INTERVAL 24 HOUR)
  AND c.valuenum BETWEEN 0.21 AND 1.0 AND c.error IS DISTINCT FROM 1
GROUP BY cs.icustay_id
"""

gcs_df  = client.query(gcs_query).to_dataframe()
fio2_df = client.query(fio2_query).to_dataframe()
print(f'GCS available:  {len(gcs_df)} stays')
print(f'FiO2 available: {len(fio2_df)} stays')


In [ ]:
def sofa_respiratory(pao2, fio2):
    fio2 = 0.21 if pd.isna(fio2) or fio2 == 0 else fio2
    pf = pao2 / fio2 if not pd.isna(pao2) else np.nan
    if pd.isna(pf): return 0
    if pf >= 400: return 0
    if pf >= 300: return 1
    if pf >= 200: return 2
    if pf >= 100: return 3
    return 4

def sofa_coagulation(plt_):
    if pd.isna(plt_): return 0
    if plt_ >= 150: return 0
    if plt_ >= 100: return 1
    if plt_ >= 50:  return 2
    if plt_ >= 20:  return 3
    return 4

def sofa_liver(bili):
    if pd.isna(bili): return 0
    if bili < 1.2:  return 0
    if bili < 2.0:  return 1
    if bili < 6.0:  return 2
    if bili < 12.0: return 3
    return 4

def sofa_cardiovascular(map_val):
    if pd.isna(map_val): return 0
    return 0 if map_val >= 70 else 1

def sofa_cns(gcs):
    if pd.isna(gcs): return 0
    if gcs >= 15: return 0
    if gcs >= 13: return 1
    if gcs >= 10: return 2
    if gcs >= 6:  return 3
    return 4

def sofa_renal(creat):
    if pd.isna(creat): return 0
    if creat < 1.2:  return 0
    if creat < 2.0:  return 1
    if creat < 3.5:  return 2
    if creat < 5.0:  return 3
    return 4

sofa_df = df[['icustay_id','mortality','pao2_min','platelets_min',
              'bilirubin_mean','map_min','creatinine_max']].copy()
sofa_df = sofa_df.merge(gcs_df,  on='icustay_id', how='left')
sofa_df = sofa_df.merge(fio2_df, on='icustay_id', how='left')

sofa_df['sofa_resp']   = sofa_df.apply(lambda r: sofa_respiratory(r['pao2_min'], r['fio2_min']), axis=1)
sofa_df['sofa_coag']   = sofa_df['platelets_min'].apply(sofa_coagulation)
sofa_df['sofa_liver']  = sofa_df['bilirubin_mean'].apply(sofa_liver)
sofa_df['sofa_cardio'] = sofa_df['map_min'].apply(sofa_cardiovascular)
sofa_df['sofa_cns']    = sofa_df['gcs_min'].apply(sofa_cns)
sofa_df['sofa_renal']  = sofa_df['creatinine_max'].apply(sofa_renal)
sofa_df['sofa_total']  = sofa_df[['sofa_resp','sofa_coag','sofa_liver',
                                   'sofa_cardio','sofa_cns','sofa_renal']].sum(axis=1)

print(f'SOFA mean: {sofa_df["sofa_total"].mean():.1f}  median: {sofa_df["sofa_total"].median():.0f}  max: {sofa_df["sofa_total"].max():.0f}')
print(f'GCS available:  {sofa_df["gcs_min"].notna().sum()} / {len(sofa_df)}')
print(f'FiO2 available: {sofa_df["fio2_min"].notna().sum()} / {len(sofa_df)}')
print(sofa_df['sofa_total'].describe().round(1))


In [ ]:
test_icustay_ids = df.iloc[X_test.index]['icustay_id'].values
sofa_lookup = sofa_df.set_index('icustay_id')['sofa_total']
sofa_test   = sofa_lookup.reindex(test_icustay_ids).fillna(sofa_df['sofa_total'].median()).values

sofa_auc = roc_auc_score(y_test, sofa_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(sofa_df[sofa_df['mortality']==0]['sofa_total'],
             bins=20, alpha=0.6, color='steelblue', label='Survived', density=True)
axes[0].hist(sofa_df[sofa_df['mortality']==1]['sofa_total'],
             bins=20, alpha=0.6, color='tomato', label='Died', density=True)
axes[0].set_xlabel('SOFA Score'); axes[0].set_ylabel('Density')
axes[0].set_title('SOFA Score Distribution by Outcome', fontweight='bold')
axes[0].legend()

y_prob_test_tuned = tuned_xgb.predict_proba(X_test)[:, 1]
fpr_s, tpr_s, _ = roc_curve(y_test, sofa_test)
fpr_x, tpr_x, _ = roc_curve(y_test, y_prob_test_tuned)
axes[1].plot(fpr_s, tpr_s, color='darkorange', linewidth=2, label=f'SOFA Score (AUC={sofa_auc:.3f})')
axes[1].plot(fpr_x, tpr_x, color='steelblue', linewidth=2, label=f'XGBoost Tuned (AUC={roc_auc_score(y_test, y_prob_test_tuned):.3f})')
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC: ML Model vs SOFA Baseline', fontweight='bold')
axes[1].legend(loc='lower right')
plt.tight_layout(); plt.show()

print(f'SOFA AUC (test):       {sofa_auc:.3f}')
print(f'XGBoost Tuned AUC:     {roc_auc_score(y_test, y_prob_test_tuned):.3f}')
print(f'Improvement over SOFA: +{roc_auc_score(y_test, y_prob_test_tuned) - sofa_auc:.3f}')


2. Statistical Significance, Bootstrap AUC Confidence Intervals

In [ ]:
def bootstrap_auc_ci(y_true, y_prob, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    y_true_arr, y_prob_arr = np.array(y_true), np.array(y_prob)
    n = len(y_true_arr)
    aucs = []
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        if len(np.unique(y_true_arr[idx])) < 2: continue
        aucs.append(roc_auc_score(y_true_arr[idx], y_prob_arr[idx]))
    aucs = np.array(aucs)
    return aucs.mean(), np.percentile(aucs, 2.5), np.percentile(aucs, 97.5)

models_to_test = [
    ('Logistic Regression',   lr.predict_proba(X_test_scaled)[:, 1]),
    ('Random Forest',          rf.predict_proba(X_test)[:, 1]),
    ('XGBoost (Default)',       xgb.predict_proba(X_test)[:, 1]),
    ('XGBoost (Tuned)',         tuned_xgb.predict_proba(X_test)[:, 1]),
    ('XGBoost (Calibrated)',    calibrated_xgb.predict_proba(X_test)[:, 1]),
    ('SOFA Score (baseline)',   sofa_test),
]

print('Bootstrap AUC with 95% CI (n_boot=1000), Test Set')
print('=' * 60)
boot_results = []
for name, probs in models_to_test:
    mean_auc, lo, hi = bootstrap_auc_ci(y_test, probs)
    boot_results.append({'Model': name, 'AUC': mean_auc, 'CI_low': lo, 'CI_high': hi})
    print(f'{name:30s}  {mean_auc:.3f}  [{lo:.3f} – {hi:.3f}]')

boot_df = pd.DataFrame(boot_results)
fig, ax = plt.subplots(figsize=(9, 5))
y_pos = range(len(boot_df))
ax.barh(list(y_pos), boot_df['AUC'],
        xerr=[boot_df['AUC']-boot_df['CI_low'], boot_df['CI_high']-boot_df['AUC']],
        align='center', color='steelblue', alpha=0.7, capsize=5)
ax.set_yticks(list(y_pos)); ax.set_yticklabels(boot_df['Model'])
ax.set_xlabel('ROC-AUC')
ax.set_title('Model AUC with 95% Bootstrap CI (Test Set)', fontweight='bold')
ax.axvline(0.5, color='red', linestyle='--', linewidth=1, label='Random classifier')
ax.set_xlim(0.45, 0.95); ax.legend()
plt.tight_layout(); plt.show()


3. Final Model Comparison, Test Set

In [ ]:
from sklearn.metrics import brier_score_loss

def full_test_eval(name, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    report = classification_report(y_test, y_pred,
                                   target_names=['Survived','Died'], output_dict=True)
    return {
        'Model':       name,
        'AUC':         round(roc_auc_score(y_test, y_prob), 3),
        'Brier':       round(brier_score_loss(y_test, y_prob), 4),
        'Accuracy':    round(accuracy_score(y_test, y_pred), 3),
        'Sensitivity': round(report['Died']['recall'], 3),
        'Specificity': round(report['Survived']['recall'], 3),
        'F1 (Died)':   round(report['Died']['f1-score'], 3),
    }

summary_rows = [
    full_test_eval('Logistic Regression',  lr.predict_proba(X_test_scaled)[:, 1]),
    full_test_eval('Random Forest',         rf.predict_proba(X_test)[:, 1]),
    full_test_eval('XGBoost (Default)',      xgb.predict_proba(X_test)[:, 1]),
    full_test_eval('XGBoost (Tuned)',        tuned_xgb.predict_proba(X_test)[:, 1]),
    full_test_eval('XGBoost (Calibrated)',   calibrated_xgb.predict_proba(X_test)[:, 1]),
    full_test_eval('SOFA Score (baseline)',  sofa_test / sofa_test.max()),
]

summary_df = pd.DataFrame(summary_rows).set_index('Model')
print('Final Model Comparison, Test Set')
print('=' * 75)
print(summary_df.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(summary_df.astype(float), annot=True, fmt='.3f',
            cmap='YlGnBu', ax=ax, linewidths=0.5, annot_kws={'size': 10})
ax.set_title('Final Model Comparison, Test Set', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()
